# 用 Claude Managed Agents 构建 SRE 事件响应智能体

## 引言

生产告警在凌晨 3 点触发时，总得有人去拉取日志、找到对应的运维手册、追溯错误配置、开一个 PR 并让它通过审批。智能体可以替你完成这第一轮工作，等你坐到键盘前时修复方案已经在等待评审——只要它拿到正确的上下文，并且由人来做出最终决定。

[Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) 提供了可扩展的基础设施、沙箱和安全组件，让你轻松构建这样的系统。在本教程中你会把它们串起来：

- 一个模拟的 **PagerDuty webhook** 用一次 API 调用触发你的 Claude Managed Agent。
- 一个 **Skill** 教会智能体你团队的运维手册约定，让它知道该去哪里查找。
- 内置的 `bash`/`read`/`edit` 工具让它在沙箱中调查日志和基础设施代码。
- **自定义工具**让它可以开一个 pull request，并在合并前请求人工审批——这些调用由你的代码处理，所以由你决定"开一个 PR"实际意味着什么。
- **Anthropic Console** 自动记录每一个步骤，为你提供完整的可观测性。

下面所有内容只需 `ANTHROPIC_API_KEY` 即可运行。PagerDuty、GitHub 和 Datadog 都用本地测试数据模拟，让你专注于 Managed Agents 的各个组件；结尾一节展示如何把每个模拟替换为真实服务。

### 你将学到什么

- 上传一个 Skill 并把它挂接到 Claude Managed Agent
- 把内置工具集与由你的应用处理的自定义工具混合使用
- 从 webhook 载荷启动一个会话
- 把破坏性操作放在人工审批之后
- 在 Console 中阅读完整的会话轨迹

### 前置条件

在环境中设置 `ANTHROPIC_API_KEY`，然后安装依赖：

In [1]:
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import json
import os
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from utilities import wait_for_idle_status

load_dotenv()
client = Anthropic()
MODEL = os.getenv("COOKBOOK_MODEL", "claude-opus-4-6")
FIXTURE = Path("example_data/sre")

## 1. 上传一个运维手册技能

[**Skill**](https://platform.claude.com/docs/en/managed-agents/skills) 是一个小型文件系统包，平台以渐进披露的方式把它挂载进智能体的上下文：智能体开头只看到一行描述，只在相关内容时才读取正文。它很适合存放不该写进系统提示词的团队约定。

下面这个示例技能编码了一条规则——*改动基础设施之前先查阅运维手册*——按真实团队 playbook 的写法。你通过 Skills API 上传一次，之后在每个需要它的智能体上按 ID 引用。

In [3]:
# A real skill is usually a folder on disk (SKILL.md plus any helper
# scripts or reference docs) that you zip and upload. For this tutorial
# the SKILL.md is small enough to keep inline.
RUNBOOK_SKILL = """\
---
name: incident-runbooks
description: How to triage production incidents using the team runbooks.
---

# Incident runbooks

When an alert references a service, locate that service's recent logs
and identify the failure signature (the repeating error class, exit
code, or status pattern).

Consult the team runbooks before proposing any fix. Runbooks are
organised by failure signature — for example `oom.md`, `5xx.md`,
`latency.md`. Each one lists the triage steps for that class of
failure and the configuration that usually needs to change.

Any fix to infrastructure code must be opened as a pull request that
cites the runbook you followed. Do not patch live resources directly.
"""

skill = client.beta.skills.create(
    display_title="incident-runbooks",
    files=[("incident-runbooks/SKILL.md", RUNBOOK_SKILL.encode(), "text/markdown")],
)
print(f"skill: {skill.id} (version {skill.latest_version})")

skill: skill_01WPWHALbtEVBUWG6mHa7Tna (version 1775588716519983)


## 2. 创建智能体

智能体的 `tools` 列表组合了三类能力：

- [﷐0﷑](https://platform.claude.com/docs/en/managed-agents/tools) — 在沙箱*内部*运行的内置 `bash`、`read`、`grep`、`edit` 等工具。智能体用这些工具进行调查。
- 步骤 1 中的运维手册 **Skill**。
- 三个**自定义工具** — `open_pull_request`、`request_approval` 和 `merge_pull_request` — 智能体可以调用，但*由你的应用*执行。它们是智能体触达沙箱之外系统的途径，也是你把人放进决策回路的方式。

系统提示词只包含角色人格和工作流。告警本身作为第一个用户事件到达，所以同一个智能体能处理任意事件。

In [4]:
SRE_SYSTEM_PROMPT = """\
You are an on-call SRE agent. Each user message is a PagerDuty alert
payload. Triage it to root cause and ship the minimal safe fix.

The session workspace contains the recent logs, the infrastructure
repo, and the team runbooks for the alerting service. Explore it to
find what you need.

Workflow for every alert:
1. Read the logs and identify the failure signature.
2. Find the root cause in the infrastructure repo, save a copy of the
   original file, edit it in place, then produce a unified diff with
   `diff -u`.
3. open_pull_request(title, body, diff) with the fix.
4. request_approval(summary) and wait for the human's decision.
5. Only if the result is "approved", merge_pull_request(pr_number).
   Otherwise stop and report.

Never call merge_pull_request unless request_approval returned
"approved". Keep the fix minimal — do not refactor unrelated config.
"""

agent = client.beta.agents.create(
    name="cookbook-sre-responder",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/sre-incident-responder"},
    model=MODEL,
    system=SRE_SYSTEM_PROMPT,
    skills=[{"type": "custom", "skill_id": skill.id, "version": skill.latest_version}],
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        },
        {
            "type": "custom",
            "name": "open_pull_request",
            "description": "Open a pull request against the infra repo with the proposed fix.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "body": {"type": "string"},
                    "diff": {"type": "string", "description": "Unified diff of the change."},
                },
                "required": ["title", "body", "diff"],
            },
        },
        {
            "type": "custom",
            "name": "request_approval",
            "description": "Ask the on-call human to approve the proposed PR before merging.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                },
                "required": ["summary"],
            },
        },
        {
            "type": "custom",
            "name": "merge_pull_request",
            "description": "Merge an approved pull request.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "pr_number": {"type": "integer"},
                },
                "required": ["pr_number"],
            },
        },
    ],
)
print(f"agent: {agent.id} v{agent.version}")

agent: agent_011CZpw3Y76Vu4t2j2QEosVa v1


## 3. 创建环境并挂载数据

智能体的工作区需要三样东西才能进行调查：近期服务日志、基础设施仓库和团队运维手册。通过 Files API 逐个上传，并把它们列为 `resources`，这样它们会在每个会话中挂载到系统提示词预期的路径上。一个 `limited` 网络的云环境就够了，因为智能体只需要自己的文件系统。

为了让本笔记本仅用 `ANTHROPIC_API_KEY` 就能运行，基础设施"仓库"是一个单一清单，其中 `memory: 128Mi` 限额设置得过低。在生产环境中，你会把这个上传替换为一个 `github_repository` 资源，把真实仓库直接克隆进沙箱：

```python
{
    "type": "github_repository",
    "url": "https://github.com/your-org/infra",
    "authorization_token": os.environ["GITHUB_TOKEN"],
    "checkout": {"type": "branch", "name": "main"},
    "mount_path": "infra",
}
```

In [5]:
env = client.beta.environments.create(
    name="cookbook-sre-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)


def upload(path: Path, mime: str) -> str:
    with path.open("rb") as f:
        return client.beta.files.upload(file=(path.name, f, mime)).id


log_id = upload(FIXTURE / "logs/checkout-svc.log", "text/plain")
manifest_id = upload(FIXTURE / "infra/k8s/checkout-deploy.yaml", "text/yaml")
runbook_id = upload(FIXTURE / "runbooks/oom.md", "text/markdown")

RESOURCES = [
    {"type": "file", "file_id": log_id, "mount_path": "logs/checkout-svc.log"},
    {"type": "file", "file_id": manifest_id, "mount_path": "infra/k8s/checkout-deploy.yaml"},
    {"type": "file", "file_id": runbook_id, "mount_path": "runbooks/oom.md"},
]
print(f"environment: {env.id}")

environment: env_01R6hmJkd6BhpPotXnoC7rqU


## 4. 处理事件告警

下面的处理函数是唯一需要部署的函数——一个 Flask 或 FastAPI 路由，由你的告警系统在事件发生时调用。它创建一个引用智能体和环境的会话，挂载数据，并把告警 JSON 作为第一个 `user.message` 事件发送。这个示例使用 [PagerDuty V3 webhook](https://developer.pagerduty.com/docs/webhooks-overview) 载荷，但任何能 POST JSON 的寻呼系统都以相同方式工作；这里你直接用测试数据调用处理函数。

In [6]:
def handle_pagerduty_webhook(payload: dict) -> str:
    incident = payload["event"]["data"]
    session = client.beta.sessions.create(
        environment_id=env.id,
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        resources=RESOURCES,
        title=f"[{incident['service']['summary']}] {incident['title']}",
    )
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            }
        ],
    )
    return session.id


with (FIXTURE / "alert.json").open() as f:
    alert = json.load(f)

session_id = handle_pagerduty_webhook(alert)
print(f"session: {session_id}")

session: sesn_011CZpw3gtC691y7qmaLNmLM


## 5. 处理智能体的自定义工具调用

这里是内置工具和你的自定义工具汇合的地方。智能体的 `read`/`bash`/`edit` 调用在容器上执行，在事件日志中显示为 `agent.tool_use` — 那就是调查过程，你只需打印出来。但当智能体调用你的某个自定义工具时，会话进入 `idle` 且 `stop_reason.type == "requires_action"`，等待*你的应用*用一个 `user.custom_tool_result` 响应。

下面的循环轮询 `events.list`，通过写入一个本地列表就地回答 `open_pull_request` 和 `merge_pull_request` — 那就是 GitHub 的模拟 — 但在 `request_approval` 到达时**返回**，因为那一项需要人工。

在生产环境中，"需要人工"通常意味着*把它发到 Slack*：把智能体的摘要发到值班频道，带一个 **Approve** 按钮，并在有人点击时把结果送回去。[﷐0﷑ cookbook](slack_data_bot.ipynb) 展示了这部分的 Bolt 接线；这里你在下一个单元格中就地审批，以保持笔记本自包含。

In [7]:
prs: list[dict] = []
pending_approvals: list[dict] = []
seen_events: set[str] = set()


def handle_custom_tool(name: str, args: dict) -> dict:
    if name == "open_pull_request":
        n = len(prs) + 1
        prs.append({"number": n, "merged": False, **args})
        print(f"\n── PR #{n}: {args['title']} ──")
        return {"pr_number": n, "url": f"mock://infra/pull/{n}"}
    if name == "merge_pull_request":
        prs[args["pr_number"] - 1]["merged"] = True
        return {"merged": True}
    raise ValueError(f"unhandled tool {name}")


def run_until_approval_or_end(session_id: str) -> str | None:
    """Poll the session's event log, servicing custom tools, until either
    a request_approval call arrives (return its event_id so the caller
    can respond) or the agent ends its turn (return None)."""
    custom_calls: dict[str, object] = {}
    responded: set[str] = set()
    while True:
        idle_stop = None
        for ev in client.beta.sessions.events.list(session_id):
            if ev.id in seen_events:
                continue
            seen_events.add(ev.id)
            if ev.type == "agent.message":
                for block in ev.content:
                    if block.type == "text":
                        print(block.text, end="")
            elif ev.type == "agent.tool_use":
                print(f"\n  [{ev.name}]")
            elif ev.type == "agent.custom_tool_use":
                custom_calls[ev.id] = ev
                print(f"\n→ {ev.name}")
            elif ev.type == "session.status_idle":
                idle_stop = ev.stop_reason
            elif ev.type == "session.status_terminated":
                return None
        if idle_stop is None:
            time.sleep(1.0)
            continue
        if idle_stop.type == "end_turn":
            return None
        if idle_stop.type == "requires_action":
            for event_id in idle_stop.event_ids:
                if event_id in responded:
                    continue
                call = custom_calls[event_id]
                if call.name == "request_approval":
                    pending_approvals.append({"event_id": event_id, **call.input})
                    return event_id
                result = handle_custom_tool(call.name, call.input)
                client.beta.sessions.events.send(
                    session_id,
                    events=[
                        {
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(result)}],
                        }
                    ],
                )
                responded.add(event_id)
        time.sleep(1.0)


approval_event_id = run_until_approval_or_end(session_id)

I'll help you triage this production incident. The alert shows checkout-svc pods are crash-looping with 7 restarts in 5 minutes. Let me start by exploring the workspace to find logs and the infrastructure configuration.
  [bash]
Let me check the session directory for logs and infrastructure files:
  [bash]
Perfect! Let me examine the logs first to identify the failure signature:
  [read]
Perfect! I've identified the failure signature. The logs show a clear **OutOfMemoryError** pattern:

**Failure Signature:**
- Service starts and warms pricing cache (14,092 entries)
- Heap memory grows rapidly: 101MB → 118MB → 121MB (against 128MB limit)
- `pricing.recompute` function attempts allocation during garbage collection pause (412ms GC pause observed)
- Container gets OOMKilled (exit 137) after ~2 minutes
- Service restarts and immediately repeats the cycle

Now let me examine the deployment configuration to identify the root cause:
  [read]
**Root Cause Analysis:**

The deployment has memory

智能体已经读取了日志，通过技能把 `OOMKilled` 特征匹配到 `runbooks/oom.md`，在 `infra/k8s/checkout-deploy.yaml` 中找到了 128Mi 的内存限额，修改了它，开了一个 PR，现在正在等你。这就是会落到你的 `#oncall` Slack 频道的消息：

In [8]:
pr = prs[0]
print(pr["body"], "\n")
print(pr["diff"])
print("\n" + "─" * 60)
print("APPROVAL REQUESTED:", pending_approvals[0]["summary"])

## Issue
checkout-svc pods are in a CrashLoopBackOff state due to OutOfMemoryError. The service consistently crashes after ~2 minutes with 7 restarts in the last 5 minutes.

## Root Cause
The deployment had memory limits set to 128Mi, which is insufficient for the pricing cache operation:
- Pricing cache warms with 14,092 entries during startup
- Heap pressure builds to 118-121MB (92-94% of limit) within 90 seconds
- pricing.recompute fails to allocate 8MB, causing OOMKilled (exit 137)
- Service restarts and repeats the cycle

## Fix
Increase memory allocation to provide adequate headroom:
- **Memory request:** 128Mi → 256Mi
- **Memory limit:** 128Mi → 512Mi

This provides 4x headroom for the pricing cache and normal operations while remaining resource-efficient (512Mi limit is standard for Java/similar workloads with caching).

## Verification
The fix addresses the immediate OOMKilled pattern in logs and aligns memory resources with the actual cache size and operational requirements. 

## 6. 审批并让智能体合并

把 `"approved"` 作为 `request_approval` 的结果送回去。智能体恢复执行，调用 `merge_pull_request`，然后结束它这一轮。在 Slack 版本中，这个发送动作发生在你的按钮点击处理函数里——载荷完全相同。

In [9]:
client.beta.sessions.events.send(
    session_id,
    events=[
        {
            "type": "user.custom_tool_result",
            "custom_tool_use_id": approval_event_id,
            "content": [{"type": "text", "text": json.dumps({"decision": "approved"})}],
        }
    ],
)

run_until_approval_or_end(session_id)
print(f"\n\nPR #{pr['number']} merged: {prs[0]['merged']}")

Great! The fix has been approved. Now merging:
→ merge_pull_request
## ✅ Incident Resolved

**Summary:**
- **Status:** MERGED (PR #1)
- **Failure:** checkout-svc crash-loop with OOMKilled (exit 137)
- **Root Cause:** Memory limit of 128Mi was insufficient for pricing cache (14,092 entries)
- **Fix Applied:** 
  - Memory request: 128Mi → 256Mi
  - Memory limit: 128Mi → 512Mi

**What Happened:**
1. Service loaded 14k+ pricing cache entries during startup
2. Heap grew to 118-121MB within ~2 minutes (92-94% of 128Mi limit)
3. pricing.recompute failed to allocate 8MB, triggering OutOfMemoryError
4. Container was OOMKilled and restarted, repeating the cycle

**Expected Outcome:**
With 512Mi limit and 256Mi request, the service will have sufficient memory headroom for:
- Pricing cache operations
- Normal request processing
- JVM garbage collection pauses
- No more CrashLoopBackOff

The deployment update will trigger a rolling restart of the 3 replicas, allowing pods to spawn with the new memo

## 7. 在 Console 中回顾这次运行

因为调查是作为 Managed Agents 会话运行的，上面每一步——文件读取、`bash` diff、清单编辑、三次自定义工具调用，以及你发送的审批——都作为事件持久化在会话上。在 [Console](https://platform.claude.com/) 中打开它，位于 **Managed Agents → Sessions**，即可获得完整的审计轨迹，无需额外埋点：

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/sre/console_session.png" alt="Console session view for the incident-response run" width="700" />

### 清理

归档会话和你创建的资源。

In [10]:
wait_for_idle_status(client, session_id)
client.beta.sessions.archive(session_id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.skills.versions.delete(skill.latest_version, skill_id=skill.id)
client.beta.skills.delete(skill.id)
print("archived")

archived


## 下一步：生产接线

三处替换就能把这套东西从笔记本搬到值班响应链路。

**在 Slack 中审批。**当 `request_approval` 到达时，用 Block Kit 按钮把它发到值班频道，并从动作处理函数把 `user.custom_tool_result` 送回去。[﷐2﷑ cookbook](slack_data_bot.ipynb) 讲了 Bolt 应用的搭建；与审批相关的部分很小：

```python
def post_for_approval(session_id, event_id, summary):
    slack.client.chat_postMessage(
        channel=ONCALL_CHANNEL,
        text=summary,
        blocks=[
            {"type": "section", "text": {"type": "mrkdwn", "text": summary}},
            {"type": "actions", "elements": [
                {"type": "button", "text": {"type": "plain_text", "text": "Approve"},
                 "action_id": "approve", "value": f"{session_id}:{event_id}"},
                {"type": "button", "text": {"type": "plain_text", "text": "Reject"},
                 "action_id": "reject", "value": f"{session_id}:{event_id}"},
            ]},
        ],
    )

@slack.action("approve")
def on_approve(ack, body):
    ack()
    session_id, event_id = body["actions"][0]["value"].split(":")
    client.beta.sessions.events.send(
        session_id,
        events=[{"type": "user.custom_tool_result",
                 "custom_tool_use_id": event_id,
                 "content": [{"type": "text",
                              "text": json.dumps({"decision": "approved"})}]}],
    )
```

要彻底去掉轮询循环，请在 `session.requires_action` 上注册一个 Console webhook — 平台会在智能体暂停的那一刻调用你的端点，你从那里发到 Slack。

**用 GitHub 替代模拟。**去掉 `open_pull_request` / `merge_pull_request` 自定义工具，改为给智能体 GitHub MCP server，令牌存放在 vault 中，这样它永远不会出现在你的代码里。[﷐2﷑](CMA_operate_in_production.ipynb) 讲解了按用户划分的凭据。

```python
agent = client.beta.agents.create(
    ...,
    mcp_servers=[{"type": "url", "name": "github",
                  "url": "https://api.githubcopilot.com/mcp/"}],
    tools=[{"type": "agent_toolset_20260401", ...},
           {"type": "mcp_toolset", "server_name": "github"},
           {"type": "custom", "name": "request_approval", ...}],
)
session = client.beta.sessions.create(..., vault_ids=[github_vault.id])
```

**用实时日志替代测试数据。**把 `DD_API_KEY` / `DD_APP_KEY` 通过环境配置传进去，让智能体从 `bash` 里 `curl` Datadog Logs API，而不是读取挂载的文件。

> 另参见 [Agent SDK 站点可靠性智能体](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb)，它用本地 Agent SDK 而不是托管的 Managed Agents 运行时解决了同一个问题。


## 你学到了什么

- 从任意外部事件触发一个会话 — PagerDuty webhook 的一次 API 调用就启动了整次运行。
- 挂接一个 **Skill**，把团队的约定交给智能体。
- 用 **resources** 挂载数据：代码用 `github_repository`，日志和运维手册用 `file`。
- 用 **自定义工具**回调你的应用，并通过 `requires_action` 把操作放在人工审批之后。
- 在 **Console** 会话视图中免费获得完整的审计轨迹。

把模拟件替换为 GitHub MCP、一个 Slack 审批按钮和实时日志，这套东西就能用于值班响应了。